In [21]:
import openai, json, requests

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"
"""
/movies
/movies/:id
/movies/:id/credits
/movies/:id/similar
"""

client = openai.OpenAI()
messages = []

In [ ]:
def get_popular_movies():
    return requests.get(f"{BASE_URL}/movies").text


def get_movie_details(id):
    return requests.get(f"{BASE_URL}/movies/{id}/credits").text


def get_similar_movies(id):
    return requests.get(f"{BASE_URL}/movies/{id}/similar").text


FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies,
}

In [23]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of currently popular movies.",
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The movie ID.",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Get movies similar to a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The movie ID.",
                    }
                },
                "required": ["id"],
            },
        },
    },
]

In [ ]:
from openai.types.chat import ChatCompletionMessage


def process_ai_response(message: ChatCompletionMessage):

    if message.tool_calls:

        # record request
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        # call(run)
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            # print(f"[Calling]: {function_name} with {arguments} 호출")

            try:
                arguments = json.loads(arguments)  # to python obj
            except json.JSONDecodeError:
                arguments = {}

            args_str = ", ".join(str(v) for v in arguments.values())
            print(f"[AGENT]: [{function_name}({args_str}) 호출]")

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)
            # print(f"[Ran]: {function_name} with {arguments} for a result of {result}")

            # record result
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result,
                }
            )

        # exit loop
        call_ai()

    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"[AGENT]: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-5-nano",
        messages=messages,
        tools=TOOLS,
    )

    # response.choices[0].message.tool_calls

    process_ai_response(response.choices[0].message)

In [25]:
while True:
    message = input("Send a message: ")

    if message == "exit" or message == "quit" or message == "q":
        break

    else:
        messages.append({"role": "user", "content": message})
        print(f"[USER]: {message}")

        call_ai()

[USER]: 지금 인기 있는 영화 알려줘
[AGENT]: [get_popular_movies() 호출]
[AGENT]: 지금 인기 있는 영화 상위 5편을 간단히 정리해 드릴게요.

1) Shelter
- 개봉일: 2026-01-28
- 줄거리: 은둔 생활을 하는 남자가 폭풍에서 소년을 구조했고, 과거의 적들로부터 그를 지키려는 이야기가 벌어집니다.
- 평점: 6.8/10
- 간단한 비고: 액션/범죄/스릴러 분위기의 신작

2) The Bluff
- 개봉일: 2026-02-17
- 줄거리: 외딴 섬에서 평온하던 삶이 전 선장의 복귀로 흔들리자, 전 해적 출신의 숙련된 인물이 가족을 지키기 위해 맹공을 펼칩니다.
- 평점: 약 5.99/10
- 간단한 비고: 액션/스릴러

3) Mercy
- 개봉일: 2026-01-20
- 줄거리: 가까운 미래, 아내를 살해한 혐의로 재판에 선 형사. 그는 AI 재판관 앞에서 무죄를 증명하려 시도합니다.
- 평점: 7.15/10
- 간단한 비고: SF/추리 스릴러

4) Pose
- 개봉일: 2026-02-25
- 줄거리: 고립된 저택에서 전 애인, 집착하는 팬, 뮤즈 후보와 함께 보낸 주말 동안 예술적 영감과 파국이 교차합니다.
- 평점: 8.0/10
- 간단한 비고: 심리/드라마

5) Hellfire
- 개봉일: 2026-02-05
- 줄거리: 과거를 쓸고 지나온 떠돌이와 마을 사람들이 범죄 보스의 grip 아래 놓인 작은 마을에서 서로 돕고 싸웁니다.
- 평점: 7.2/10
- 간단한 비고: 액션/범죄

추가로 더 보고 싶은 작품이 있나요? 특정 영화의 상세 정보나 비슷한 분위기의 추천도 바로 드릴게요. 영화 제목이나 TMDB ID를 알려주시면 상세 정보를 조회해 드립니다.
[USER]: Mercy에 대해 더 알려줘
[AGENT]: [get_movie_details(1236153) 호출]
[AGENT]: Mercy(머시)에 대해 지금까지 확인된 정보를 정리해 드릴게요.

- 기본 정보
  - 제목: Mercy
  - 장르